# FSDL Day3 - 데이터 탐색 노트북

## 목적
- KLUE-YNAT 데이터셋의 분포와 특성 파악
- TF-IDF 베이스라인 구축
- 이후 BERT Fine-tuning을 위한 인사이트 도출

> **FSDL Day1 원칙**: "모델 학습 전에 데이터와 충분히 친해져라."
> "데이터를 10배 더 들여다보는 것이 아키텍처 탐색보다 효과적이다."

In [ ]:
# RunPod 환경: 절대경로 설정
import os, sys

# notebooks/ 폴더에서 실행되므로 프로젝트 루트는 한 단계 위
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
os.chdir(PROJECT_ROOT)  # 작업 디렉토리를 프로젝트 루트로 변경
sys.path.insert(0, PROJECT_ROOT)

print(f'프로젝트 루트: {PROJECT_ROOT}')

In [ ]:
from datasets import load_dataset
from collections import Counter
import pandas as pd

# ── 데이터 로드 ──
dataset = load_dataset('klue', 'ynat')

print(f'학습 데이터: {len(dataset["train"]):,}건')
print(f'검증 데이터: {len(dataset["validation"]):,}건')

# ── 레이블 이름 확인 ──
label_names = dataset['train'].features['label'].names
print(f'\n카테고리 ({len(label_names)}개): {label_names}')

In [ ]:
# ── 샘플 5개 직접 확인 ──
# FSDL Day1: "데이터와 가까워지기 — 직접 눈으로 보는 것이 최고의 탐색"
print('=== 샘플 5개 ===\n')
for i in range(5):
    sample = dataset['train'][i]
    label_name = label_names[sample['label']]
    print(f'[{label_name}] {sample["title"]}')

print('\n→ 관찰: 제목은 얼마나 긴가요? 카테고리 판단이 애매한 케이스가 있나요?')

In [ ]:
# ── 클래스별 분포 확인 ──
label_counts = Counter(dataset['train']['label'])
print('── 클래스별 분포 ──')
for label_id, count in sorted(label_counts.items()):
    ratio = count / len(dataset['train'])
    bar = '█' * int(ratio * 50)
    print(f'  {label_names[label_id]:6s} | {count:5d} ({ratio:5.1%}) {bar}')

print('\n→ 클래스 불균형이 심하면 Macro F1을 사용해야 합니다.')

In [ ]:
# ── 제목 길이 분포 ──
lengths = [len(t) for t in dataset['train']['title']]
print(f'── 제목 길이 ──')
print(f'  최소: {min(lengths)}자')
print(f'  최대: {max(lengths)}자')
print(f'  평균: {sum(lengths)/len(lengths):.0f}자')
print(f'  90th percentile: {sorted(lengths)[int(len(lengths)*0.9)]}자')

print('\n→ max_length=128이면 대부분의 제목을 커버합니다.')

In [ ]:
# ── TF-IDF 베이스라인 ──
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score

vectorizer = TfidfVectorizer(max_features=10000)
X_train = vectorizer.fit_transform(dataset['train']['title'])
X_val = vectorizer.transform(dataset['validation']['title'])
y_train = dataset['train']['label']
y_val = dataset['validation']['label']

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_val)
macro_f1 = f1_score(y_val, y_pred, average='macro')

print(f'\n베이스라인 Macro F1: {macro_f1:.4f}  ← 이 수치를 실행 계획서에 기록하세요')
print()
print(classification_report(y_val, y_pred, target_names=label_names))